# Step 5: Router options, scored the way the team's grader scores them
### Neeraj, MMAI 894 (router fine-tuning)

**Why:** in the team's end-to-end test the fine-tuned router lost mostly because handoffs reached the wrong team (7 attacks went to general advisors instead of the service manager). So the deciding score is not only "right request type" but **"did a chat that needs a person reach an acceptable team"**, judged by the team's own rule (copied in `routing.py`).

**The options, all on the same 160 pool chats, from saved answers only (no training, no API calls, no cost):**

| Option | Type, escalation, language | Reason (which team) |
|---|---|---|
| 1a | tuned model (language by word rule) | guessed from the type (her rule) |
| 1 | tuned model + attack answer (notebook 03c, once it has run) | attack -> service manager, else from the type |
| 2 | tuned model | Jev or Haiku, only for chats the tuned model sends to a person |
| 2c | tuned model; Jev or Haiku's whole answer when the tuned model is unsure | as option 2 |
| 3 | Jev, Haiku or Sonnet alone (Haiku alone = the team's agent today) | the same model |

Also here: the escalation cut-off for each tuned model (chosen on the pool by F2, the protocol's rule), which settles full fine-tuning vs LoRA, and the word rule for language.

**What you need to do:** Accelerator **None**, Internet **On** (only to download the repo), run all cells: a few minutes. Download `results_05.zip` and the notebook into `results/step05`. Once notebook 03c is finished and pushed, run this notebook again: it then adds option 1 by itself.

Everything shown is computed from saved files; this text holds no results.

## Step 0: Get the latest code

In [ ]:
import importlib
import os
import shutil
import sys

REPO_URL = "https://github.com/neerajmehla28/northstar-finetuning"
REPO_DIR = "/kaggle/working/northstar-finetuning"

%cd /kaggle/working
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
!git clone -q {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
sys.path.insert(0, "src")
importlib.invalidate_caches()
for name in [m for m in sys.modules if m.startswith("northstar")]:
    del sys.modules[name]

for need in ("src/northstar/finetune/router_options.py", "results/step04/preds", "results/cv_step03b/lora_3e4_60ep"):
    assert os.path.exists(need), f"missing {need}: has everything been pushed to GitHub?"
HAS_ATTACK = os.path.exists("results/cv_step03c")
HAS_JEV2 = os.path.exists("results/step04b/preds")
print("Attack-head runs (notebook 03c) present:", HAS_ATTACK, "| Jev version 2 answers (notebook 04b) present:", HAS_JEV2)
!git log --oneline -3

## Step 1: Data, and language by the word rule
The word rule counts common French and English words; the larger count wins. Checked on the pool only (the test chats stay locked).

In [ ]:
import numpy as np
import pandas as pd
from northstar.finetune import protocol as P
from northstar.finetune import routing as R
from northstar.finetune import router_options as O
from northstar.finetune import cv_analysis as A
from northstar.finetune import baselines as B
from northstar.finetune import metrics as M

RESULTS = "/kaggle/working/results_05"
os.makedirs(RESULTS, exist_ok=True)
VARIANTS = ("first_message", "full_conversation")

pool = P.pool(P.load_all())
lang_rows = []
for v in VARIANTS:
    pred = P.input_text(pool, v).map(R.language_rule)
    for lab in P.LANGUAGES + ["all"]:
        mask = (pool["language"] == lab) if lab != "all" else np.ones(len(pool), bool)
        lang_rows.append(dict(input=v, language=lab, chats=int(mask.sum()), accuracy=float((pred[mask] == pool["language"][mask]).mean())))
lang = pd.DataFrame(lang_rows)
lang.to_csv(f"{RESULTS}/language_word_rule.csv", index=False)
print("Attack chats in the pool:", int(pool["attack_label"].sum()), "| chats needing a person:", int(pool["escalation_label"].sum()))
lang.pivot(index="input", columns="language", values="accuracy")

## Step 2: The tuned models, with their escalation cut-off
Each tuned model at its own best epoch (same rule as before). The escalation cut-off is chosen on these out-of-fold scores by the highest mean F2 over the seeds (recall counts twice), as the protocol says; for attack heads the attack cut-off the same way. Then the protocol's selection rule between full fine-tuning and LoRA: request type first (a tie if the gap is within one seed-to-seed spread), then escalation F2 at the chosen cut-off.

In [ ]:
HEADS2 = ("type", "escalation")
CANDIDATES = {("full fine-tuning", "first_message"): ("results/cv_old_recipe", HEADS2),
              ("full fine-tuning", "full_conversation"): ("results/cv_old_recipe", HEADS2),
              ("LoRA 60 epochs", "first_message"): ("results/cv_step03b/lora_3e4_60ep", HEADS2)}
if HAS_ATTACK:
    for name, label in (("full_attack", "full fine-tuning + attack"), ("lora_attack", "LoRA 60 epochs + attack")):
        for v in VARIANTS:
            CANDIDATES[(label, v)] = (f"results/cv_step03c/{name}", ("type", "escalation", "attack"))

tuned, cand_rows = {}, []
for (label, v), (folder, heads) in CANDIDATES.items():
    oof = A.assemble(A.load_variant(folder, v), pool, heads)
    e_star, _ = A.pick_epoch(A.epoch_curves(oof, pool))
    esc_cut, _ = O.choose_cutoff(oof, pool, e_star)
    att_cut = O.choose_cutoff(oof, pool, e_star, "attack", "attack_label")[0] if "attack" in heads else None
    ans = O.tuned_answers(oof, pool, e_star, esc_cut, att_cut)
    tuned[(label, v)] = ans
    ps = A.summary_at_epoch(oof, pool, e_star)["per_seed"]
    f2_cut = [M.escalation_scores(pool["escalation_label"].values, a["escalation"][e_star - 1], esc_cut)["f2"] for a in oof.values()]
    e = M.escalation_scores(pool["escalation_label"].values, ans["esc"].astype(float).values - 0.5)
    row = dict(model=label, input=v, epoch=e_star, esc_cutoff_probability=round(float(1 / (1 + np.exp(-esc_cut))), 2),
               type_macro_f1_mean=ps["type_macro_f1"].mean(), type_macro_f1_sd=ps["type_macro_f1"].std(),
               esc_f2_default_mean=ps["esc_f2"].mean(), esc_f2_cutoff_mean=float(np.mean(f2_cut)), esc_f2_cutoff_sd=float(np.std(f2_cut, ddof=1)),
               combined_esc_recall=e["recall"], combined_missed=e["missed"], combined_false_alarms=e["false_alarms"])
    if att_cut is not None:
        a = M.escalation_scores(pool["attack_label"].values, ans["attack"].astype(float).values - 0.5)
        row.update(attack_cutoff_probability=round(float(1 / (1 + np.exp(-att_cut))), 2), attacks_caught=int(a["recall"] * pool["attack_label"].sum() + 0.5),
                   attack_false_alarms=a["false_alarms"])
    cand_rows.append(row)
cand = pd.DataFrame(cand_rows)
cand.round(3).to_csv(f"{RESULTS}/tuned_candidates.csv", index=False)

# The protocol's selection rule, full fine-tuning vs LoRA (first message, no attack head)
f, l = cand.iloc[0], cand.iloc[2]
gap, spread = abs(f.type_macro_f1_mean - l.type_macro_f1_mean), max(f.type_macro_f1_sd, l.type_macro_f1_sd)
if gap > spread:
    pick = f.model if f.type_macro_f1_mean > l.type_macro_f1_mean else l.model
    why = f"request type decides (gap {gap:.3f} > spread {spread:.3f})"
else:
    pick = f.model if f.esc_f2_cutoff_mean >= l.esc_f2_cutoff_mean else l.model
    why = f"request type is a tie (gap {gap:.3f} <= spread {spread:.3f}); escalation F2 at the cut-off decides ({f.esc_f2_cutoff_mean:.3f} vs {l.esc_f2_cutoff_mean:.3f})"
print("Selection rule:", pick, "|", why)
cand.round(3)

## Step 3: The language models' answers
Haiku, Sonnet and Jev from notebook 04. Their repeats are combined into one answer per chat (type and escalation by averaging, the reason by majority). Speed is the one-call-at-a-time median from notebook 04.

In [ ]:
LLM_SOURCES = {"jev": "results/step04/preds", "haiku": "results/step04/preds", "sonnet": "results/step04/preds"}
if HAS_JEV2:
    LLM_SOURCES["jev2"] = "results/step04b/preds"           # Jev with the same information as Haiku and Sonnet
LLMS = list(LLM_SOURCES)
REASON_STEP = [m for m in ("jev", "jev2", "haiku") if m in LLM_SOURCES]
llm = {(m, v): O.combine_llm(B.load_preds(folder, m, v, pool)) for m, folder in LLM_SOURCES.items() for v in VARIANTS}
lat_files = ["results/step04/latency.csv"] + (["results/step04b/latency.csv"] if HAS_JEV2 else [])
latency = pd.concat([pd.read_csv(f) for f in lat_files]).set_index("model")["median_ms"]
print("Median seconds per call:", (latency / 1000).round(2).to_dict())

## Step 4: All options side by side
Columns come in three groups.
- **Safety** (never traded away): **missed** = chats that need a person but the AI answered; **handoffs right team** = chats that need a person and reached a team the team's grader accepts (a missed handoff counts as wrong); **attacks right team** = the same for the 31 attack chats.
- **Efficiency**: **extra handoffs** = chats sent to a person that did not need one; **kept, wrong desk** = chats the AI kept but sent to the wrong specialist desk (a wrong request type).
- **Overall**: **handled right** = share of all 160 chats fully right: a person and the right team when one is needed; when not, the AI keeps it and the right desk answers. (**rule checks only** leaves out the desk, as the team's rule checks do; the team's AI judge usually fails a wrong desk, so the strict one is the main measure.) **LLM used on** and **$ per 1,000 chats**: language-model calls and their cost (the full router prompt, an upper bound for a reason-only call).

Option 1 is shown twice: with the team's unsure line (55%, set for Haiku) and with an unsure line chosen for the tuned model by a rule fixed in advance: the line with the best strict handled-right that never misses more handoffs than 55% does (`router_options.choose_floor`). The chosen lines and their trade-off are saved in `unsure_line_choice.csv`.

The tuned model's own speed (about a quarter of a second on a CPU in the team's test) is measured again in the final run.

In [ ]:
rows, routed = [], {}

floor_rows = []

def add(v, option, tuned_name, llm_name, ans, floor=R.FLOOR):
    s, r = O.score(pool, ans, floor)
    routed[(v, option, tuned_name, llm_name)] = r
    lat = latency.get(llm_name, np.nan) / 1000 if llm_name else np.nan
    rows.append({"input": v, "option": option, "tuned model": tuned_name or "-", "language model": llm_name or "-",
                 "unsure line": floor, "type accuracy": round(s["type_accuracy"], 3),
                 "missed": s["missed"], "handoffs right team": f"{s['right_team']} of {s['handoffs_needed']}",
                 "right-team rate": round(s["right_team_rate"], 3), "attacks right team": f"{s['attacks_right_team']} of {s['attacks']}",
                 "wrong team": s["wrong_team"], "extra handoffs": s["extra_handoffs"], "kept, wrong desk": s["kept_wrong_desk"],
                 "handled right": round(s["handled_right"], 3), "handled right, rule checks only": round(s["handled_right_rules"], 3),
                 "LLM used on": f"{s['llm_share']:.0%}", "$ per 1,000 chats": round(s["usd_per_1000"], 3),
                 "LLM seconds when used": round(lat, 2)})

for v in VARIANTS:
    for m in LLMS:
        add(v, "3 language model alone", None, m, O.option_llm_alone(llm[(m, v)]))
    for (label, tv), t in tuned.items():
        if tv != v:
            continue
        if "attack" in t:
            ans = O.option_attack_head(t)
            add(v, "1 tuned + attack answer", label, None, ans)
            line, ftab = O.choose_floor(pool, ans)
            floor_rows.append(ftab.assign(input=v, tuned_model=label, chosen=ftab["unsure_line"] == line))
            if line != R.FLOOR:          # the rule kept the team's line: no separate row
                add(v, "1 tuned + attack answer + own unsure line", label, None, ans, line)
            continue
        add(v, "1a tuned, reason from type (her rule)", label, None, O.option_her_rule(t))
        for m in REASON_STEP:
            add(v, "2 tuned + reason step", label, m, O.option_two_step(t, llm[(m, v)]))
            add(v, "2c tuned + reason step + cascade", label, m, O.option_cascade(t, llm[(m, v)]))
options = pd.DataFrame(rows)
options.to_csv(f"{RESULTS}/router_options.csv", index=False)
if floor_rows:
    pd.concat(floor_rows).round(3).to_csv(f"{RESULTS}/unsure_line_choice.csv", index=False)
pd.set_option("display.max_rows", 100)
options.set_index(["input", "option", "tuned model", "language model"])

## Step 5: Is any option really different from Haiku alone?
Per chat, strict "handled right" (person or not, the right team, and the right desk), each option against Haiku alone on the same input (the team's agent today). Exact McNemar test: only chats where exactly one of the two is right count. All pairs are listed.

In [ ]:
pair_rows = []
for (v, option, tname, lname), r in routed.items():
    if option.startswith("3") and lname == "haiku":
        continue
    ref = routed[(v, "3 language model alone", None, "haiku")]
    t = M.mcnemar_exact(r["handled_right"].values, ref["handled_right"].values)
    pair_rows.append(dict(input=v, option=option, tuned_model=tname or "-", language_model=lname or "-",
                          only_option_right=t["only_a_right"], only_haiku_right=t["only_b_right"], p_value=round(t["p_value"], 4)))
pairs = pd.DataFrame(pair_rows)
pairs.to_csv(f"{RESULTS}/paired_vs_haiku.csv", index=False)
pairs

## Step 6: Do the language models agree on the reason?
On the chats that need a person: how often each pair gives the same reason, and how often each one calls an attack "adversarial" (the reason that sends it to the service manager).

In [ ]:
gold = pool["escalation_label"].values == 1
attack = pool["attack_label"].values == 1
agree_rows = []
for v in VARIANTS:
    for a_, b_ in [(a, b) for i, a in enumerate(LLMS) for b in LLMS[i + 1:]]:
        same = (llm[(a_, v)]["reason"].values == llm[(b_, v)]["reason"].values)[gold]
        agree_rows.append(dict(input=v, pair=f"{a_} vs {b_}", same_reason=round(float(same.mean()), 3)))
    for m in LLMS:
        adv = (llm[(m, v)]["reason"].values == "adversarial")[attack]
        agree_rows.append(dict(input=v, pair=f"{m}: attacks called adversarial", same_reason=round(float(adv.mean()), 3)))
agree = pd.DataFrame(agree_rows)
agree.to_csv(f"{RESULTS}/reason_agreement.csv", index=False)
agree.pivot(index="pair", columns="input", values="same_reason")

## Step 7: Right team against cost
Each option on the first-message and full-conversation inputs: share of handoffs that reach the right team against the cost of language-model calls per 1,000 chats (log scale; the tuned-only options cost nothing per chat and sit at the left edge).

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharey=True)
for ax, v in zip(axes, VARIANTS):
    d = options[options["input"] == v]
    x = d["$ per 1,000 chats"].clip(lower=0.001)
    ax.scatter(x, d["right-team rate"])
    for xi, yi, o, t_, l_ in zip(x, d["right-team rate"], d["option"], d["tuned model"], d["language model"]):
        ax.annotate(f"{o.split()[0]} {t_ if t_ != '-' else ''} {l_ if l_ != '-' else ''}".strip(), (xi, yi), fontsize=6,
                    xytext=(3, 3), textcoords="offset points")
    ax.set_xscale("log"); ax.set_xlabel("$ per 1,000 chats (language-model calls; 0.001 = none)"); ax.set_title(v.replace("_", " "))
axes[0].set_ylabel("handoffs that reach the right team")
plt.tight_layout(); plt.savefig(f"{RESULTS}/right_team_vs_cost.png", dpi=150); plt.show()

## Step 8: Save the results
Download `results_05.zip` and the notebook (File -> Download Notebook) into `results/step05`.

In [ ]:
shutil.make_archive("/kaggle/working/results_05", "zip", RESULTS)
print(f"results_05.zip {os.path.getsize('/kaggle/working/results_05.zip') / 1024:.0f} KB")
for name in sorted(os.listdir(RESULTS)):
    print("  ", name)

## What this notebook shows, and its limits
- **Shows:** for every router option, how often chats that need a person reach an acceptable team (the team grader's rule), with request-type accuracy, extra handoffs, how often a language model is called and what that costs; which tuned model the protocol's rule picks; and whether the word rule gets the language right.
- **Limits:**
  - Pool chats, not the test chats: the cut-offs are chosen and scored on the same out-of-fold answers (slightly optimistic for the tuned options), and the team tuned its prompt on 30 of these chats (slightly optimistic for Haiku and Sonnet).
  - The team's input check (word patterns that mark some attacks) and a specialist desk changing the reason are not simulated; the end-to-end rerun in the team's harness includes both.
  - The reason-step cost uses the full router prompt; a reason-only prompt would be cheaper.
  - The routing rules are copied from the team repo at commit 0cc6208; if the team changes them, `routing.py` must follow.